SK_ID_PREV：對應的歷史貸款編號（一筆現有貸款可能連到多筆過往貸款）。

SK_ID_CURR：目前這筆樣本中的貸款編號。

MONTHS_BALANCE：相對於申請日的月份（-1 為最新月）。

CNT_INSTALMENT：當月記錄中的分期間數（可隨時間變動）。

CNT_INSTALMENT_FUTURE：當前仍未繳的分期數。

SK_DPD：該月的逾期天數。

SK_DPD_DEF：同樣是逾期天數，但忽略低額欠款。

NAME_CONTRACT_STATUS_Active / Completed / Signed / Other：這四個欄位為 one-hot 編碼，代表該客戶在整個 POS_CASH 數據中，契約狀態落在對應類別的月份筆數（例：NAME_CONTRACT_STATUS_Active 為 Active 狀態的月數）。

In [2]:
import pandas as pd

pos_cash = pd.read_csv('home-credit-default-risk/POS_CASH_balance.csv')

In [3]:
pos_cash.columns

Index(['SK_ID_PREV', 'SK_ID_CURR', 'MONTHS_BALANCE', 'CNT_INSTALMENT',
       'CNT_INSTALMENT_FUTURE', 'NAME_CONTRACT_STATUS', 'SK_DPD',
       'SK_DPD_DEF'],
      dtype='object')

In [4]:
missing_rate_percent = (pos_cash.isnull().mean() * 100).sort_values(ascending=False)
print(missing_rate_percent)

CNT_INSTALMENT_FUTURE    0.260835
CNT_INSTALMENT           0.260675
SK_ID_PREV               0.000000
SK_ID_CURR               0.000000
MONTHS_BALANCE           0.000000
NAME_CONTRACT_STATUS     0.000000
SK_DPD                   0.000000
SK_DPD_DEF               0.000000
dtype: float64


In [5]:
status_counts = pos_cash['NAME_CONTRACT_STATUS'].value_counts()
pos_cash['NAME_CONTRACT_STATUS'] = pos_cash['NAME_CONTRACT_STATUS'].apply(
    lambda x: x if status_counts[x] >= 10000 else 'Other'
)
print(pos_cash['NAME_CONTRACT_STATUS'].value_counts())

NAME_CONTRACT_STATUS
Active       9151119
Completed     744883
Signed         87260
Other          18096
Name: count, dtype: int64


In [6]:
## 這兩個合在一起可以表示欠的比例

months_with_dpd = (
    pos_cash.groupby('SK_ID_CURR')['SK_DPD_DEF']
    .apply(lambda x: (x > 0).sum())
    .reset_index(name='pos_months_with_dpd')
)

monthly_records = (
    pos_cash.groupby('SK_ID_CURR')
    .size()
    .reset_index(name='pos_cash_record_count')
)

In [7]:
categorical_cols = ['NAME_CONTRACT_STATUS']
pos_cash = pd.get_dummies(pos_cash, columns=categorical_cols, drop_first=False)

agg_dict = {
    'MONTHS_BALANCE': ['mean'],
    'CNT_INSTALMENT': ['mean'],
    'CNT_INSTALMENT_FUTURE': ['mean'], 
    'SK_DPD': ['max'], 
    'SK_DPD_DEF': ['max'], 
}

for col in pos_cash.columns:
    if col.startswith('NAME_CONTRACT_STATUS_'):
        agg_dict[col] = ['sum']

pos_cash_agg = pos_cash.groupby('SK_ID_CURR').agg(agg_dict)
pos_cash_agg.columns = [f'{col}_{stat}' for col, stat in pos_cash_agg.columns]
pos_cash_agg = pos_cash_agg.reset_index()

pos_cash_agg = pos_cash_agg.merge(months_with_dpd, on='SK_ID_CURR', how='left')
pos_cash_agg = pos_cash_agg.merge(monthly_records, on='SK_ID_CURR', how='left')

pos_cash_agg.head()

,SK_ID_CURR,MONTHS_BALANCE_mean,CNT_INSTALMENT_mean,CNT_INSTALMENT_FUTURE_mean,SK_DPD_max,SK_DPD_DEF_max,NAME_CONTRACT_STATUS_Active_sum,NAME_CONTRACT_STATUS_Completed_sum,NAME_CONTRACT_STATUS_Other_sum,NAME_CONTRACT_STATUS_Signed_sum,pos_months_with_dpd,pos_cash_record_count
0,100001,-72.555556,4.000000,1.444444,7,7,7,2,0,0,1,9
1,100002,-10.000000,24.000000,15.000000,0,0,19,0,0,0,0,19
2,100003,-43.785714,10.107143,5.785714,0,0,26,2,0,0,0,28
3,100004,-25.500000,3.750000,2.250000,0,0,3,1,0,0,0,4
4,100005,-20.000000,11.700000,7.200000,0,0,9,1,0,1,0,11


In [8]:
pd.DataFrame.to_csv(pos_cash_agg, "transformed_data/_pos_cash.csv") 